## **Title:** Consistency and Faithfulness Analysis of XAI Methods for Brain Tumor MRI Classification Without Pixel-Level Ground Truth


## 0. Environment checks

In [1]:
import sys, platform
import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv2D, BatchNormalization, Activation,MaxPooling2D, Flatten, Dense, Dropout)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras import layers

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="keras.src.models.functional")


print('Python:', sys.version)
print('Platform:', platform.platform())
print('TensorFlow:', tf.__version__)

# Optional: GPU info (Kaggle)
try:
    from subprocess import check_output
    print(check_output(['nvidia-smi','-L']).decode())
except Exception as e:
    print('GPU info not available:', e)


2026-01-17 14:35:01.632255: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768660501.784759      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768660501.828651      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768660502.182425      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768660502.182480      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768660502.182483      55 computation_placer.cc:177] computation placer alr

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
TensorFlow: 2.19.0
GPU 0: Tesla P100-PCIE-16GB (UUID: GPU-10b8e1cc-04c9-5d0b-34df-0459a931bb1e)



## 1. Imports and global configuration

In [2]:
import os, json, random
from pathlib import Path

import numpy as np
import cv2
from tqdm import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Local project utilities (if running from repo root)
# from xai_validator import xai_validate_model

SEED = 101
VAL_FRAC = 0.15
IMAGE_SIZE = 224
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
EPOCHS = 100
NUM_CLASSES = 4
LABELS = ['glioma','meningioma','notumor','pituitary']

def seed_everything(seed: int = 101):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

seed_everything(SEED)

# Reproducibility notes:
# - Some GPU ops are nondeterministic; we still fix seeds and record splits.


## 2. Dataset paths and file listing

In [3]:
# Kaggle path default
DATA_ROOT = Path('/kaggle/input/brain-tumor-mri-dataset')
TRAIN_DIR = DATA_ROOT / 'Training'
TEST_DIR  = DATA_ROOT / 'Testing'

def list_images(class_dir: Path):
    exts = {'.png','.jpg','.jpeg','.bmp'}
    files = [str(class_dir / f) for f in os.listdir(class_dir) if Path(f).suffix.lower() in exts]
    files.sort()
    return files

train_paths = {c: list_images(TRAIN_DIR / c) for c in LABELS}
test_paths  = {c: list_images(TEST_DIR  / c) for c in LABELS}

print('Training counts:')
for c in LABELS:
    print(f'  {c}: {len(train_paths[c])}')
print('Testing counts:')
for c in LABELS:
    print(f'  {c}: {len(test_paths[c])}')

# Flatten
train_files = [(p, c) for c in LABELS for p in train_paths[c]]
test_files  = [(p, c) for c in LABELS for p in test_paths[c]]

print('Total train:', len(train_files))
print('Total test :', len(test_files))


Training counts:
  glioma: 1321
  meningioma: 1339
  notumor: 1595
  pituitary: 1457
Testing counts:
  glioma: 300
  meningioma: 306
  notumor: 405
  pituitary: 300
Total train: 5712
Total test : 1311


## 3. Preprocessing

In [4]:
def preprocess_image(path: str, image_size: int = 224):
    # Load grayscale
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f'Failed to read: {path}')

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)

    # Bilateral filter
    img = cv2.bilateralFilter(img, d=2, sigmaColor=50, sigmaSpace=50)

    # Pseudo-color
    img = cv2.applyColorMap(img, cv2.COLORMAP_BONE)

    # Resize
    img = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_AREA)

    # Convert to float32
    img = img.astype(np.float32)

    # Per-image z-score
    mean = np.mean(img, axis=(0,1,2), keepdims=True)
    std  = np.std(img, axis=(0,1,2), keepdims=True)
    img = (img - mean) / (std + 1e-8)

    return img

label_to_idx = {c:i for i,c in enumerate(LABELS)}

def one_hot(label: str, num_classes: int = 4):
    y = np.zeros((num_classes,), dtype=np.float32)
    y[label_to_idx[label]] = 1.0
    return y


## 4. Create Train/Val split

In [5]:
# Build arrays of file paths and numeric labels for stratification
X_paths = np.array([p for p,_ in train_files])
y_str   = np.array([c for _,c in train_files])
y_idx   = np.array([label_to_idx[c] for c in y_str])

X_tr_paths, X_val_paths, y_tr_idx, y_val_idx = train_test_split(
    X_paths, y_idx,
    test_size=VAL_FRAC,
    random_state=SEED,
    stratify=y_idx
)

# Save split manifest for reproducibility
split_manifest = {
    'seed': SEED,
    'val_frac': VAL_FRAC,
    'train_paths': X_tr_paths.tolist(),
    'val_paths': X_val_paths.tolist(),
    'label_map': LABELS
}
Path('artifacts').mkdir(exist_ok=True)
with open('artifacts/splits.json','w') as f:
    json.dump(split_manifest, f, indent=2)

print('Train size:', len(X_tr_paths))
print('Val size  :', len(X_val_paths))

# Build locked test arrays (never used during training)
X_test_paths = np.array([p for p,_ in test_files])
y_test_idx   = np.array([label_to_idx[c] for _,c in test_files])
print('Test size :', len(X_test_paths))


Train size: 4855
Val size  : 857
Test size : 1311


## 5. Build tf.data pipelines

In [6]:
AUTOTUNE = tf.data.AUTOTUNE

def load_tf(path, y_idx):
    path = path.numpy().decode('utf-8')
    img = preprocess_image(path, IMAGE_SIZE)
    y = tf.one_hot(y_idx, depth=NUM_CLASSES, dtype=tf.float32)
    return img, y

def tf_wrapper(path, y_idx):
    img, y = tf.py_function(load_tf, inp=[path, y_idx], Tout=[tf.float32, tf.float32])
    img.set_shape((IMAGE_SIZE, IMAGE_SIZE, 3))
    y.set_shape((NUM_CLASSES,))
    return img, y

train_ds = tf.data.Dataset.from_tensor_slices((X_tr_paths, y_tr_idx))
val_ds   = tf.data.Dataset.from_tensor_slices((X_val_paths, y_val_idx))
test_ds  = tf.data.Dataset.from_tensor_slices((X_test_paths, y_test_idx))

train_ds = train_ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
train_ds = train_ds.map(tf_wrapper, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(tf_wrapper, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(tf_wrapper, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)


I0000 00:00:1768660515.578659      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


## 6. Model definition

In [7]:
def build_custom_cnn(input_shape=(224,224,3), num_classes=4):
    inputs = keras.Input(shape=input_shape)

    # Conv Block 1
    x = layers.Conv2D(32, 3, padding='same', kernel_regularizer=keras.regularizers.l2(0.001))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.MaxPooling2D()(x)

    # Conv Block 2
    x = layers.Conv2D(64, 3, padding='same', kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.MaxPooling2D()(x)

    # Conv Block 3
    x = layers.Conv2D(128, 3, padding='same', kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.MaxPooling2D()(x)

    # Conv Block 4
    x = layers.Conv2D(256, 3, padding='same', kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.MaxPooling2D()(x)

    # Conv Block 5
    x = layers.Conv2D(512, 3, padding='same', kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)
    x = layers.MaxPooling2D()(x)

    # Flatten + Classification Head
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='swish', kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='swish', kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='propose')

model = build_custom_cnn((IMAGE_SIZE, IMAGE_SIZE, 3), NUM_CLASSES)
model.summary()


Model: "propose"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 14, 14, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 14, 14, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 8,028,740 (30.63 MB)

 Trainable params: 8,026,756 (30.62 MB)

 Non-trainable params: 1,984 (7.75 KB)

## 7. Training

In [8]:
METRICS = [
    keras.metrics.CategoricalAccuracy(name='accuracy'),
    keras.metrics.Precision(name='precision'),
    keras.metrics.Recall(name='recall')
]

model.compile(
    loss='categorical_crossentropy',
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    metrics=METRICS
)

ckpt_path = 'artifacts/best_model.keras'
checkpoint = ModelCheckpoint(ckpt_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_accuracy', factor=0.1, patience=5, verbose=1)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stopping, reduce_lr],
    verbose=1
)


Epoch 1/100


I0000 00:00:1768660522.649454     132 service.cc:152] XLA service 0x7e3b7401a980 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768660522.649494     132 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1768660523.653875     132 cuda_dnn.cc:529] Loaded cuDNN version 91002


  3/304 ━━━━━━━━━━━━━━━━━━━━ 10s 35ms/step - accuracy: 0.3299 - loss: 8.8081 - precision: 0.3188 - recall: 0.2535   

I0000 00:00:1768660530.676582     132 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


304/304 ━━━━━━━━━━━━━━━━━━━━ 0s 127ms/step - accuracy: 0.4559 - loss: 4.9316 - precision: 0.4957 - recall: 0.3765
Epoch 1: val_accuracy improved from -inf to 0.48191, saving model to artifacts/best_model.keras
304/304 ━━━━━━━━━━━━━━━━━━━━ 59s 155ms/step - accuracy: 0.4561 - loss: 4.9260 - precision: 0.4961 - recall: 0.3765 - val_accuracy: 0.4819 - val_loss: 2.0607 - val_precision: 0.6734 - val_recall: 0.3874 - learning_rate: 0.0010
Epoch 2/100
303/304 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.6085 - loss: 1.8580 - precision: 0.7361 - recall: 0.4727
Epoch 2: val_accuracy improved from 0.48191 to 0.60093, saving model to artifacts/best_model.keras
304/304 ━━━━━━━━━━━━━━━━━━━━ 35s 115ms/step - accuracy: 0.6087 - loss: 1.8571 - precision: 0.7362 - recall: 0.4729 - val_accuracy: 0.6009 - val_loss: 1.6583 - val_precision: 0.8037 - val_recall: 0.4107 - learning_rate: 0.0010
Epoch 3/100
303/304 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.6770 - loss: 1.4829 - precision: 0.7897 - rec

## 8. Final evaluation on locked test set

In [9]:
# Load best weights (selected on validation only)
model = keras.models.load_model('artifacts/best_model.keras')

# Evaluate
test_metrics = model.evaluate(test_ds, verbose=1)
print('Test metrics:', dict(zip(model.metrics_names, test_metrics)))

# Detailed report
# Collect predictions
y_true = []
y_pred = []
for xb, yb in test_ds:
    p = model.predict(xb, verbose=0)
    y_true.extend(np.argmax(yb.numpy(), axis=1).tolist())
    y_pred.extend(np.argmax(p, axis=1).tolist())

print(classification_report(y_true, y_pred, target_names=LABELS, digits=4))
cm = confusion_matrix(y_true, y_pred)
print('Confusion matrix:', cm)

# Save results
with open('artifacts/test_metrics.json','w') as f:
    json.dump({
        'metrics_names': model.metrics_names,
        'metrics_values': [float(x) for x in test_metrics],
        'classification_report': classification_report(y_true, y_pred, target_names=LABELS, digits=4, output_dict=True),
        'confusion_matrix': cm.tolist()
    }, f, indent=2)


82/82 ━━━━━━━━━━━━━━━━━━━━ 10s 104ms/step - accuracy: 0.9427 - loss: 0.3917 - precision: 0.9426 - recall: 0.9420
Test metrics: {'loss': 0.308132529258728, 'compile_metrics': 0.9618611931800842}
              precision    recall  f1-score   support

      glioma     0.9789    0.9300    0.9538       300
  meningioma     0.9309    0.9248    0.9279       306
     notumor     0.9853    0.9901    0.9877       405
   pituitary     0.9460    0.9933    0.9691       300

    accuracy                         0.9619      1311
   macro avg     0.9603    0.9596    0.9596      1311
weighted avg     0.9622    0.9619    0.9617      1311

Confusion matrix: [[279  18   0   3]
 [  6 283   6  11]
 [  0   1 401   3]
 [  0   2   0 298]]


## 9. XAI evaluation

This section computes faithfulness and agreement metrics **without pixel-level ground truth**. Overlap metrics quantify agreement between explanation methods, not clinical correctness.

In [10]:

from pathlib import Path
import sys
import subprocess
import numpy as np

# ---- 1) Repo configuration ----
GITHUB_URL = "https://github.com/LalithK90/DL-Approaches-for-Brain-Tumor-Detection-using-MRI-images-LLM-intergration.git"
REPO_NAME  = "DL-Approaches-for-Brain-Tumor-Detection-using-MRI-images-LLM-intergration"
REPO_DIR   = Path("/kaggle/working") / REPO_NAME

print("CWD:", Path.cwd())
print("Expected REPO_DIR:", REPO_DIR)

# ---- 2) Clone repo if not present ----
if not REPO_DIR.exists():
    print("Repo not found. Cloning from GitHub...")
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(REPO_DIR)], check=True)
else:
    # Repo dir exists; check it has files
    if not any(REPO_DIR.iterdir()):
        print("Repo directory exists but is empty. Re-cloning...")
        subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL, str(REPO_DIR)], check=True)

# Show top-level repo files to confirm
print("\nRepo top-level files:", [p.name for p in REPO_DIR.iterdir()][:30])

# ---- 3) Ensure xai_validator.py exists ----
XAI_FILE = REPO_DIR / "xai_validator.py"
if not XAI_FILE.exists():
    # Try common layouts if you put code under src/
    candidates = list(REPO_DIR.rglob("xai_validator.py"))
    if candidates:
        XAI_FILE = candidates[0]
        print("Found xai_validator at:", XAI_FILE)
    else:
        raise FileNotFoundError(
            "xai_validator.py not found in repo after cloning. "
            "Confirm it exists in GitHub and is pushed."
        )

# ---- 4) Add repo (or file parent) to PYTHONPATH ----
sys.path.insert(0, str(XAI_FILE.parent))
print("Import path inserted:", XAI_FILE.parent)

# ---- 5) Import ----
from xai_validator import xai_validate_model
print("Imported xai_validate_model from:", XAI_FILE)

# ---- 6) Artifacts dir ----
ARTIFACTS_DIR = REPO_DIR / "artifacts"
(ARTIFACTS_DIR / "metrics").mkdir(parents=True, exist_ok=True)

# ---- 7) Pick last conv layer ----
CONV_LAYER = None
for layer in reversed(model.layers):
    if "conv" in (layer.name or "").lower():
        CONV_LAYER = layer.name
        break
if CONV_LAYER is None:
    raise ValueError("No conv layer found. Set CONV_LAYER manually.")
print("Using conv layer:", CONV_LAYER)

# ---- 8) Ensure labels are one-hot ----
y_for_xai = y_test_idx  # your locked test labels
if not isinstance(y_for_xai, np.ndarray):
    y_for_xai = np.array(y_for_xai)

if y_for_xai.ndim == 1:
    num_classes = int(np.max(y_for_xai)) + 1
    y_for_xai = np.eye(num_classes, dtype=np.float32)[y_for_xai.astype(int)]
elif y_for_xai.ndim != 2:
    raise ValueError(f"Unsupported y_test shape: {y_for_xai.shape}")

print("y_for_xai shape:", y_for_xai.shape)



x_test = []
for xb, _ in test_ds:
    x_test.append(xb.numpy())
x_test = np.concatenate(x_test, axis=0)
# ---- 9) Run XAI evaluation (locked test set) ----
xai_out = ARTIFACTS_DIR / "metrics" / "xai_metrics.txt"

xai_validate_model(
    model=model,
    images=x_test,
    labels=y_for_xai,
    conv_layer=CONV_LAYER,
    max_samples=30,
    title="XAI quantitative metrics (top_percent=15)",
    output_file=str(xai_out),
)

print("Saved XAI summary to:", xai_out.resolve())


CWD: /kaggle/working
Expected REPO_DIR: /kaggle/working/DL-Approaches-for-Brain-Tumor-Detection-using-MRI-images-LLM-intergration
Repo not found. Cloning from GitHub...


Cloning into '/kaggle/working/DL-Approaches-for-Brain-Tumor-Detection-using-MRI-images-LLM-intergration'...
Updating files: 100% (147/147), done.



Repo top-level files: ['braintumoridentificationapp', 'docs', 'XAI_validation', 'LICENSE', 'brain_tumor_identification_api', 'brain tumor dataset', '.gitignore', 'model_training_notebook', '.git', 'readme.md', 'data collection sheet', '.whitesource', 'start_apps.sh', 'stop_apps.sh']
Found xai_validator at: /kaggle/working/DL-Approaches-for-Brain-Tumor-Detection-using-MRI-images-LLM-intergration/XAI_validation/xai_validator.py
Import path inserted: /kaggle/working/DL-Approaches-for-Brain-Tumor-Detection-using-MRI-images-LLM-intergration/XAI_validation
Imported xai_validate_model from: /kaggle/working/DL-Approaches-for-Brain-Tumor-Detection-using-MRI-images-LLM-intergration/XAI_validation/xai_validator.py
Using conv layer: conv2d_4
y_for_xai shape: (1311, 4)


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

...

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]


XAI quantitative metrics (top_percent=15)
Dice (GC vs LIME)             : 0.242908
IoU (GC vs LIME)              : 0.143497
Dice (GC vs Saliency)         : 0.212722
IoU (GC vs Saliency)          : 0.124444
Comprehensiveness             : 0.208709
Sufficiency                   : 0.211396
Deletion AUC                  : 9.068819
Insertion AUC                 : 12.899836
Brier Score                   : 0.982005
Softmax Entropy               : 0.484412
Prediction Margin             : 0.660023
MC Dropout Variance           : 0.009955
Top-3 Prob (3rd)              : 0.045768
Top-3 Prob (2nd)              : 0.140311
Top-3 Prob (1st)              : 0.800334
Randomized Weights Corr.      : 0.344851

Saved XAI summary to: /kaggle/working/DL-Approaches-for-Brain-Tumor-Detection-using-MRI-images-LLM-intergration/artifacts/metrics/xai_metrics.txt
